In [2]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus


In [3]:
load_dotenv()

user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")
database = os.getenv("DB_NAME")

engine = create_engine(
    f"mysql+pymysql://{user}:{quote_plus(password)}@{host}:{port}/{database}",
    echo=False,
    future=True
)


In [4]:
TABLE = "mental_health"

def fetch_one(sql):
    with engine.connect() as conn:
        r = conn.execute(text(sql)).mappings().first()
        return dict(r) if r else {}

def fetch_all(sql):
    with engine.connect() as conn:
        return [dict(x) for x in conn.execute(text(sql)).mappings().all()]



### **Check Data Is OK**

In [17]:
sql = f"""
SELECT
  COUNT(*) AS total_rows,

  SUM(depression = 1) AS depressed_count,
  ROUND(SUM(depression = 1) * 100.0 / COUNT(*), 2) AS depressed_pct,

  SUM(depression = 0) AS not_depressed_count,
  ROUND(SUM(depression = 0) * 100.0 / COUNT(*), 2) AS not_depressed_pct

FROM {TABLE};
"""
print(fetch_one(sql))


{'total_rows': 27853, 'depressed_count': Decimal('16300'), 'depressed_pct': Decimal('58.52'), 'not_depressed_count': Decimal('11553'), 'not_depressed_pct': Decimal('41.48')}


In [18]:
# Quick Missing Values
sql = f"""
SELECT
  SUM(id IS NULL) AS miss_id,
  SUM(gender IS NULL OR gender = '') AS miss_gender,
  SUM(age IS NULL) AS miss_age,
  SUM(city IS NULL OR city = '') AS miss_city,
  SUM(degree IS NULL OR degree = '') AS miss_degree,
  SUM(degree_category IS NULL OR degree_category = '') AS miss_degree_category,

  SUM(academic_pressure IS NULL) AS miss_academic_pressure,
  SUM(cgpa IS NULL) AS miss_cgpa,
  SUM(study_satisfaction IS NULL) AS miss_study_satisfaction,

  SUM(sleep_duration IS NULL OR sleep_duration = '') AS miss_sleep_duration,
  SUM(dietary_habits IS NULL OR dietary_habits = '') AS miss_dietary_habits,

  SUM(suicidal_thoughts IS NULL OR suicidal_thoughts = '') AS miss_suicidal_thoughts,
  SUM(work_study_hours IS NULL) AS miss_work_study_hours,
  SUM(financial_stress IS NULL) AS miss_financial_stress,
  SUM(family_history_mental_illness IS NULL OR family_history_mental_illness = '') 
      AS miss_family_history,

  SUM(red_flag IS NULL) AS miss_red_flag,
  SUM(wellness IS NULL OR wellness = '') AS miss_wellness,
  SUM(support_priority IS NULL OR support_priority = '') AS miss_support_priority,
  SUM(depression IS NULL) AS miss_depression
FROM {TABLE};
"""

print(fetch_one(sql))


{'miss_id': Decimal('0'), 'miss_gender': Decimal('0'), 'miss_age': Decimal('0'), 'miss_city': Decimal('0'), 'miss_degree': Decimal('0'), 'miss_degree_category': Decimal('0'), 'miss_academic_pressure': Decimal('0'), 'miss_cgpa': Decimal('0'), 'miss_study_satisfaction': Decimal('0'), 'miss_sleep_duration': Decimal('0'), 'miss_dietary_habits': Decimal('0'), 'miss_suicidal_thoughts': Decimal('0'), 'miss_work_study_hours': Decimal('0'), 'miss_financial_stress': Decimal('0'), 'miss_family_history': Decimal('0'), 'miss_red_flag': Decimal('0'), 'miss_wellness': Decimal('0'), 'miss_support_priority': Decimal('0'), 'miss_depression': Decimal('0')}


##### **Q1. Do we have any duplicate id values (should be zero)?**

In [ ]:
sql = f"""
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT id) AS distinct_ids,
  (COUNT(*) - COUNT(DISTINCT id)) AS duplicate_row_count
FROM {TABLE};
"""
q1 = fetch_one(sql)
print(q1)

if q1.get("duplicate_row_count", 0) > 0:
    sql_dups = f"""
    SELECT id, COUNT(*) AS cnt
    FROM {TABLE}
    GROUP BY id
    HAVING COUNT(*) > 1
    ORDER BY cnt DESC, id ASC
    LIMIT 25;
    """
    print("\nDuplicate IDs (top 25):")
    for row in fetch_all(sql_dups):
        print(row)
else:
    print("No duplicate IDs found.")


{'total_rows': 27853, 'distinct_ids': 27853, 'duplicate_row_count': 0}
No duplicate IDs found.


##### **Q2. Are all scale columns valid: `academic_pressure`, `financial_stress`, `study_satisfaction` strictly within 1–5?**


In [ ]:
sql = f"""
SELECT
  MIN(academic_pressure) AS ap_min,
  MAX(academic_pressure) AS ap_max,
  SUM(CASE WHEN academic_pressure IS NULL OR academic_pressure < 1 OR academic_pressure > 5 THEN 1 ELSE 0 END) AS ap_invalid,

  MIN(financial_stress) AS fs_min,
  MAX(financial_stress) AS fs_max,
  SUM(CASE WHEN financial_stress IS NULL OR financial_stress < 1 OR financial_stress > 5 THEN 1 ELSE 0 END) AS fs_invalid,

  MIN(study_satisfaction) AS ss_min,
  MAX(study_satisfaction) AS ss_max,
  SUM(CASE WHEN study_satisfaction IS NULL OR study_satisfaction < 1 OR study_satisfaction > 5 THEN 1 ELSE 0 END) AS ss_invalid
FROM {TABLE};
"""
q2 = fetch_one(sql)
print(q2)

total_invalid = (q2.get("ap_invalid", 0) + q2.get("fs_invalid", 0) + q2.get("ss_invalid", 0))
print("Total invalid scale rows across these 3 columns (counted per-column):", total_invalid)


{'ap_min': 1, 'ap_max': 5, 'ap_invalid': Decimal('0'), 'fs_min': 1, 'fs_max': 5, 'fs_invalid': Decimal('0'), 'ss_min': 1, 'ss_max': 5, 'ss_invalid': Decimal('0')}
Total invalid scale rows across these 3 columns (counted per-column): 0


##### **Q3. Are there any unexpected/invalid categories in `sleep_duration`, `dietary_habits`, `gender`, `family_history_mental_illness`, `suicidal_thoughts`?**


In [21]:
# gender
allowed = ("male", "female")

sql_invalid = f"""
SELECT SUM(
  CASE
    WHEN gender IS NULL THEN 1
    WHEN LOWER(TRIM(gender)) NOT IN ('{allowed[0]}','{allowed[1]}') THEN 1
    ELSE 0
  END
) AS invalid_rows
FROM {TABLE};
"""
sql_distinct = f"""
SELECT LOWER(TRIM(gender)) AS value, COUNT(*) AS n
FROM {TABLE}
GROUP BY LOWER(TRIM(gender))
ORDER BY n DESC, value ASC;
"""

with engine.connect() as conn:
    inv = conn.execute(text(sql_invalid)).mappings().first()["invalid_rows"]
    print("Column: gender")
    print("Invalid rows:", inv)
    print("Distinct values (value, n):")
    for r in conn.execute(text(sql_distinct)).mappings().all():
        print(dict(r))

Column: gender
Invalid rows: 0
Distinct values (value, n):
{'value': 'male', 'n': 15520}
{'value': 'female', 'n': 12333}


In [22]:
# sleep_duration
sql_invalid = f"""
SELECT SUM(
  CASE
    WHEN sleep_duration IS NULL THEN 1
    WHEN LOWER(TRIM(sleep_duration)) NOT IN (
      'less than 5 hours','5-6 hours','7-8 hours','more than 8 hours'
    ) THEN 1
    ELSE 0
  END
) AS invalid_rows
FROM {TABLE};
"""
sql_distinct = f"""
SELECT LOWER(TRIM(sleep_duration)) AS value, COUNT(*) AS n
FROM {TABLE}
GROUP BY LOWER(TRIM(sleep_duration))
ORDER BY n DESC, value ASC;
"""

with engine.connect() as conn:
    inv = conn.execute(text(sql_invalid)).mappings().first()["invalid_rows"]
    print("Column: sleep_duration")
    print("Invalid rows:", inv)
    print("Distinct values (value, n):")
    for r in conn.execute(text(sql_distinct)).mappings().all():
        print(dict(r))


Column: sleep_duration
Invalid rows: 18
Distinct values (value, n):
{'value': 'less than 5 hours', 'n': 8294}
{'value': '7-8 hours', 'n': 7335}
{'value': '5-6 hours', 'n': 6173}
{'value': 'more than 8 hours', 'n': 6033}
{'value': 'others', 'n': 18}


In [23]:
# dietary_habits
sql_invalid = f"""
SELECT SUM(
  CASE
    WHEN dietary_habits IS NULL THEN 1
    WHEN LOWER(TRIM(dietary_habits)) NOT IN ('unhealthy','moderate','healthy') THEN 1
    ELSE 0
  END
) AS invalid_rows
FROM {TABLE};
"""
sql_distinct = f"""
SELECT LOWER(TRIM(dietary_habits)) AS value, COUNT(*) AS n
FROM {TABLE}
GROUP BY LOWER(TRIM(dietary_habits))
ORDER BY n DESC, value ASC;
"""

with engine.connect() as conn:
    inv = conn.execute(text(sql_invalid)).mappings().first()["invalid_rows"]
    print("Column: dietary_habits")
    print("Invalid rows:", inv)
    print("Distinct values (value, n):")
    for r in conn.execute(text(sql_distinct)).mappings().all():
        print(dict(r))


Column: dietary_habits
Invalid rows: 12
Distinct values (value, n):
{'value': 'unhealthy', 'n': 10303}
{'value': 'moderate', 'n': 9903}
{'value': 'healthy', 'n': 7635}
{'value': 'others', 'n': 12}


In [24]:
# suicidal_thoughts
sql_invalid = f"""
SELECT SUM(
  CASE
    WHEN suicidal_thoughts IS NULL THEN 1
    WHEN LOWER(TRIM(suicidal_thoughts)) NOT IN ('yes','no') THEN 1
    ELSE 0
  END
) AS invalid_rows
FROM {TABLE};
"""
sql_distinct = f"""
SELECT LOWER(TRIM(suicidal_thoughts)) AS value, COUNT(*) AS n
FROM {TABLE}
GROUP BY LOWER(TRIM(suicidal_thoughts))
ORDER BY n DESC, value ASC;
"""

with engine.connect() as conn:
    inv = conn.execute(text(sql_invalid)).mappings().first()["invalid_rows"]
    print("Column: suicidal_thoughts")
    print("Invalid rows:", inv)
    print("Distinct values (value, n):")
    for r in conn.execute(text(sql_distinct)).mappings().all():
        print(dict(r))


Column: suicidal_thoughts
Invalid rows: 0
Distinct values (value, n):
{'value': 'yes', 'n': 17621}
{'value': 'no', 'n': 10232}


In [25]:
# family_history_mental_illness
sql_invalid = f"""
SELECT SUM(
  CASE
    WHEN family_history_mental_illness IS NULL THEN 1
    WHEN LOWER(TRIM(family_history_mental_illness)) NOT IN ('yes','no') THEN 1
    ELSE 0
  END
) AS invalid_rows
FROM {TABLE};
"""
sql_distinct = f"""
SELECT LOWER(TRIM(family_history_mental_illness)) AS value, COUNT(*) AS n
FROM {TABLE}
GROUP BY LOWER(TRIM(family_history_mental_illness))
ORDER BY n DESC, value ASC;
"""

with engine.connect() as conn:
    inv = conn.execute(text(sql_invalid)).mappings().first()["invalid_rows"]
    print("Column: family_history_mental_illness")
    print("Invalid rows:", inv)
    print("Distinct values (value, n):")
    for r in conn.execute(text(sql_distinct)).mappings().all():
        print(dict(r))


Column: family_history_mental_illness
Invalid rows: 0
Distinct values (value, n):
{'value': 'no', 'n': 14375}
{'value': 'yes', 'n': 13478}


##### **Q4. Are numeric ranges realistic and consistent (`age`, `cgpa`, `work_study_hours`), and do any values cluster suspiciously at extremes?**

In [26]:
# Range checks
sql = f"""
SELECT
  MIN(age) AS age_min,
  MAX(age) AS age_max,
  SUM(CASE WHEN age IS NULL OR age < 18 OR age > 59 THEN 1 ELSE 0 END) AS age_out_of_range,

  MIN(cgpa) AS cgpa_min,
  MAX(cgpa) AS cgpa_max,
  SUM(CASE WHEN cgpa IS NULL OR cgpa < 5.03 OR cgpa > 10.00 THEN 1 ELSE 0 END) AS cgpa_out_of_range,

  MIN(work_study_hours) AS hours_min,
  MAX(work_study_hours) AS hours_max,
  SUM(CASE WHEN work_study_hours IS NULL OR work_study_hours < 0 OR work_study_hours > 12 THEN 1 ELSE 0 END) AS hours_out_of_range
FROM {TABLE};
"""

print(fetch_all(sql))

[{'age_min': 18, 'age_max': 59, 'age_out_of_range': Decimal('0'), 'cgpa_min': Decimal('5.03'), 'cgpa_max': Decimal('10.00'), 'cgpa_out_of_range': Decimal('0'), 'hours_min': 0, 'hours_max': 12, 'hours_out_of_range': Decimal('0')}]


In [27]:
# Boundary spikes (quick sanity)
sql = f"""
SELECT
  SUM(CASE WHEN age = 18 THEN 1 ELSE 0 END) AS age_18_count,
  SUM(CASE WHEN age = 59 THEN 1 ELSE 0 END) AS age_59_count,

  SUM(CASE WHEN work_study_hours = 0 THEN 1 ELSE 0 END) AS hours_0_count,
  SUM(CASE WHEN work_study_hours = 12 THEN 1 ELSE 0 END) AS hours_12_count,

  SUM(CASE WHEN cgpa >= 9.5 THEN 1 ELSE 0 END) AS cgpa_ge_9_5_count,
  SUM(CASE WHEN cgpa <= 5.5 THEN 1 ELSE 0 END) AS cgpa_le_5_5_count
FROM {TABLE};
"""

print(fetch_all(sql))

[{'age_18_count': Decimal('1584'), 'age_59_count': Decimal('1'), 'hours_0_count': Decimal('1696'), 'hours_12_count': Decimal('3163'), 'cgpa_ge_9_5_count': Decimal('3697'), 'cgpa_le_5_5_count': Decimal('1788')}]


In [68]:
# work_study_hours distribution (0–12)
sql = f"""
SELECT work_study_hours AS hours, COUNT(*) AS n
FROM {TABLE}
GROUP BY work_study_hours
ORDER BY work_study_hours;
"""

print(fetch_all(sql))

[{'hours': 0, 'n': 1696}, {'hours': 1, 'n': 1148}, {'hours': 2, 'n': 1585}, {'hours': 3, 'n': 1466}, {'hours': 4, 'n': 1611}, {'hours': 5, 'n': 1289}, {'hours': 6, 'n': 2247}, {'hours': 7, 'n': 2001}, {'hours': 8, 'n': 2507}, {'hours': 9, 'n': 2022}, {'hours': 10, 'n': 4229}, {'hours': 11, 'n': 2889}, {'hours': 12, 'n': 3163}]


### **Know the Baseline (Problem Size)**

##### **Q5. What is the overall depression prevalence (count + %)?**

In [71]:
sql = f"""
SELECT
  COUNT(*) AS total_students,
  SUM(CASE WHEN depression = 1 THEN 1 ELSE 0 END) AS depressed_students,
  ROUND(100.0 * AVG(CASE WHEN depression = 1 THEN 1 ELSE 0 END), 2) AS depression_rate_pct
FROM {TABLE};
"""
print(fetch_one(sql))


{'total_students': 27853, 'depressed_students': Decimal('16300'), 'depression_rate_pct': Decimal('58.52')}


##### **Q6. What is the overall suicidal_thoughts prevalence (count + %)?**

In [72]:
sql = f"""
SELECT
  COUNT(*) AS total_students,
  SUM(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END) AS suicidal_yes_students,
  ROUND(
    100.0 * AVG(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END),
    2
  ) AS suicidal_yes_rate_pct
FROM {TABLE};
"""
print(fetch_one(sql))

{'total_students': 27853, 'suicidal_yes_students': Decimal('17621'), 'suicidal_yes_rate_pct': Decimal('63.26')}


##### **Q7. What is the overlap: among `depression=1`, what % have `suicidal_thoughts=yes`?**

In [28]:
sql = f"""
SELECT
  SUM(CASE WHEN depression = 1 THEN 1 ELSE 0 END) AS depressed_students,
  SUM(CASE WHEN depression = 1 AND LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END) AS depressed_and_suicidal_yes,
  ROUND(
    100.0 * SUM(CASE WHEN depression = 1 AND LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END)
    / NULLIF(SUM(CASE WHEN depression = 1 THEN 1 ELSE 0 END), 0),
    2
  ) AS pct_of_depressed_with_suicidal_yes
FROM {TABLE};
"""
print(fetch_one(sql))


{'depressed_students': Decimal('16300'), 'depressed_and_suicidal_yes': Decimal('13927'), 'pct_of_depressed_with_suicidal_yes': Decimal('85.44')}


##### **Q8. Depression rate by `gender`, `degree_category`, and age bands (18–21, 22–25, 26–30, 31+).**

In [ ]:
# By gender
sql = f"""
SELECT
  LOWER(TRIM(gender)) AS gender,
  COUNT(*) AS n,
  SUM(CASE WHEN depression = 1 THEN 1 ELSE 0 END) AS depressed,
  ROUND(100.0 * AVG(CASE WHEN depression = 1 THEN 1 ELSE 0 END), 2) AS depression_rate_pct
FROM {TABLE}
GROUP BY LOWER(TRIM(gender))
ORDER BY n DESC, gender ASC;
"""
for r in fetch_all(sql):
    print(r)


{'gender': 'male', 'n': 15520, 'depressed': Decimal('9096'), 'depression_rate_pct': Decimal('58.61')}
{'gender': 'female', 'n': 12333, 'depressed': Decimal('7204'), 'depression_rate_pct': Decimal('58.41')}


In [74]:
# By degree_category
sql = f"""
SELECT
  LOWER(TRIM(degree_category)) AS degree_category,
  COUNT(*) AS n,
  SUM(CASE WHEN depression = 1 THEN 1 ELSE 0 END) AS depressed,
  ROUND(100.0 * AVG(CASE WHEN depression = 1 THEN 1 ELSE 0 END), 2) AS depression_rate_pct
FROM {TABLE}
GROUP BY LOWER(TRIM(degree_category))
ORDER BY n DESC, degree_category ASC;
"""
for r in fetch_all(sql):
    print(r)


{'degree_category': 'ug', 'n': 13314, 'depressed': Decimal('7486'), 'depression_rate_pct': Decimal('56.23')}
{'degree_category': 'pg', 'n': 7913, 'depressed': Decimal('4211'), 'depression_rate_pct': Decimal('53.22')}
{'degree_category': 'school', 'n': 6070, 'depressed': Decimal('4297'), 'depression_rate_pct': Decimal('70.79')}
{'degree_category': 'phd', 'n': 521, 'depressed': Decimal('285'), 'depression_rate_pct': Decimal('54.70')}
{'degree_category': 'others', 'n': 35, 'depressed': Decimal('21'), 'depression_rate_pct': Decimal('60.00')}


In [75]:
# By age bands (18–21, 22–25, 26–30, 31+)
sql = f"""
SELECT
  CASE
    WHEN age BETWEEN 18 AND 21 THEN '18-21'
    WHEN age BETWEEN 22 AND 25 THEN '22-25'
    WHEN age BETWEEN 26 AND 30 THEN '26-30'
    WHEN age >= 31 THEN '31+'
    ELSE 'unknown'
  END AS age_band,
  COUNT(*) AS n,
  SUM(CASE WHEN depression = 1 THEN 1 ELSE 0 END) AS depressed,
  ROUND(100.0 * AVG(CASE WHEN depression = 1 THEN 1 ELSE 0 END), 2) AS depression_rate_pct
FROM {TABLE}
GROUP BY
  CASE
    WHEN age BETWEEN 18 AND 21 THEN '18-21'
    WHEN age BETWEEN 22 AND 25 THEN '22-25'
    WHEN age BETWEEN 26 AND 30 THEN '26-30'
    WHEN age >= 31 THEN '31+'
    ELSE 'unknown'
  END
ORDER BY
  FIELD(age_band, '18-21','22-25','26-30','31+','unknown');
"""
for r in fetch_all(sql):
    print(r)


{'age_band': '18-21', 'n': 7100, 'depressed': Decimal('5057'), 'depression_rate_pct': Decimal('71.23')}
{'age_band': '22-25', 'n': 6835, 'depressed': Decimal('4332'), 'depression_rate_pct': Decimal('63.38')}
{'age_band': '26-30', 'n': 7832, 'depressed': Decimal('4428'), 'depression_rate_pct': Decimal('56.54')}
{'age_band': '31+', 'n': 6086, 'depressed': Decimal('2483'), 'depression_rate_pct': Decimal('40.80')}


### **Validate Risk Tiers(Are `red_flag` and `wellness` usable?)**

##### **Q9. Depression rate and student count by `red_flag` score (0–7).**


In [76]:
sql = f"""
SELECT
  red_flag,
  COUNT(*) AS n,
  SUM(CASE WHEN depression = 1 THEN 1 ELSE 0 END) AS depressed,
  ROUND(100.0 * AVG(CASE WHEN depression = 1 THEN 1 ELSE 0 END), 2) AS depression_rate_pct
FROM {TABLE}
GROUP BY red_flag
ORDER BY red_flag;
"""
for r in fetch_all(sql):
    print(r)


{'red_flag': 0, 'n': 1320, 'depressed': Decimal('45'), 'depression_rate_pct': Decimal('3.41')}
{'red_flag': 1, 'n': 3962, 'depressed': Decimal('588'), 'depression_rate_pct': Decimal('14.84')}
{'red_flag': 2, 'n': 5800, 'depressed': Decimal('2170'), 'depression_rate_pct': Decimal('37.41')}
{'red_flag': 3, 'n': 6680, 'depressed': Decimal('4384'), 'depression_rate_pct': Decimal('65.63')}
{'red_flag': 4, 'n': 5693, 'depressed': Decimal('4907'), 'depression_rate_pct': Decimal('86.19')}
{'red_flag': 5, 'n': 3233, 'depressed': Decimal('3062'), 'depression_rate_pct': Decimal('94.71')}
{'red_flag': 6, 'n': 1015, 'depressed': Decimal('995'), 'depression_rate_pct': Decimal('98.03')}
{'red_flag': 7, 'n': 150, 'depressed': Decimal('149'), 'depression_rate_pct': Decimal('99.33')}


##### **Q10. Student distribution (%) across `wellness` (high/moderate/low).**


In [78]:
sql = f"""
SELECT
  LOWER(TRIM(wellness)) AS wellness,
  COUNT(*) AS n,

  SUM(CASE WHEN depression = 1 THEN 1 ELSE 0 END) AS depressed,
  ROUND(100.0 * AVG(CASE WHEN depression = 1 THEN 1 ELSE 0 END), 2) AS depression_rate_pct,

  SUM(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END) AS suicidal_yes,
  ROUND(
    100.0 * AVG(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END),
    2
  ) AS suicidal_yes_rate_pct
FROM {TABLE}
GROUP BY LOWER(TRIM(wellness))
ORDER BY n DESC, wellness ASC;
"""
for r in fetch_all(sql):
    print(r)


{'wellness': 'moderate', 'n': 12373, 'depressed': Decimal('9291'), 'depression_rate_pct': Decimal('75.09'), 'suicidal_yes': Decimal('9699'), 'suicidal_yes_rate_pct': Decimal('78.39')}
{'wellness': 'high', 'n': 11082, 'depressed': Decimal('2803'), 'depression_rate_pct': Decimal('25.29'), 'suicidal_yes': Decimal('3739'), 'suicidal_yes_rate_pct': Decimal('33.74')}
{'wellness': 'low', 'n': 4398, 'depressed': Decimal('4206'), 'depression_rate_pct': Decimal('95.63'), 'suicidal_yes': Decimal('4183'), 'suicidal_yes_rate_pct': Decimal('95.11')}


##### **Q11) For each wellness tier: student count + depression rate + suicidal rate**

In [79]:
sql = f"""
SELECT
  LOWER(TRIM(wellness)) AS wellness,
  COUNT(*) AS n,

  SUM(CASE WHEN depression = 1 THEN 1 ELSE 0 END) AS depressed,
  ROUND(100.0 * AVG(CASE WHEN depression = 1 THEN 1 ELSE 0 END), 2) AS depression_rate_pct,

  SUM(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END) AS suicidal_yes,
  ROUND(
    100.0 * AVG(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END),
    2
  ) AS suicidal_yes_rate_pct
FROM {TABLE}
GROUP BY LOWER(TRIM(wellness))
ORDER BY n DESC, wellness ASC;
"""
for r in fetch_all(sql):
    print(r)


{'wellness': 'moderate', 'n': 12373, 'depressed': Decimal('9291'), 'depression_rate_pct': Decimal('75.09'), 'suicidal_yes': Decimal('9699'), 'suicidal_yes_rate_pct': Decimal('78.39')}
{'wellness': 'high', 'n': 11082, 'depressed': Decimal('2803'), 'depression_rate_pct': Decimal('25.29'), 'suicidal_yes': Decimal('3739'), 'suicidal_yes_rate_pct': Decimal('33.74')}
{'wellness': 'low', 'n': 4398, 'depressed': Decimal('4206'), 'depression_rate_pct': Decimal('95.63'), 'suicidal_yes': Decimal('4183'), 'suicidal_yes_rate_pct': Decimal('95.11')}


##### **Q12. What % of all depressed students are captured by `red_flag ≥5` and `red_flag ≥6`?**

In [80]:
sql = f"""
SELECT
  SUM(CASE WHEN depression = 1 THEN 1 ELSE 0 END) AS depressed_total,

  SUM(CASE WHEN depression = 1 AND red_flag >= 5 THEN 1 ELSE 0 END) AS depressed_in_rf_ge_5,
  ROUND(
    100.0 * SUM(CASE WHEN depression = 1 AND red_flag >= 5 THEN 1 ELSE 0 END)
    / NULLIF(SUM(CASE WHEN depression = 1 THEN 1 ELSE 0 END), 0),
    2
  ) AS pct_depressed_captured_rf_ge_5,

  SUM(CASE WHEN depression = 1 AND red_flag >= 6 THEN 1 ELSE 0 END) AS depressed_in_rf_ge_6,
  ROUND(
    100.0 * SUM(CASE WHEN depression = 1 AND red_flag >= 6 THEN 1 ELSE 0 END)
    / NULLIF(SUM(CASE WHEN depression = 1 THEN 1 ELSE 0 END), 0),
    2
  ) AS pct_depressed_captured_rf_ge_6
FROM {TABLE};
"""
print(fetch_one(sql))


{'depressed_total': Decimal('16300'), 'depressed_in_rf_ge_5': Decimal('4206'), 'pct_depressed_captured_rf_ge_5': Decimal('25.80'), 'depressed_in_rf_ge_6': Decimal('1144'), 'pct_depressed_captured_rf_ge_6': Decimal('7.02')}


### **Find Main Drivers (Simple Gradients)**


##### **Q13. Depression rate by `academic_pressure` (1–5).**

In [81]:
sql = f"""
SELECT
  academic_pressure AS level,
  COUNT(*) AS n,

  SUM(CASE WHEN depression = 1 THEN 1 ELSE 0 END) AS depressed,
  ROUND(100.0 * AVG(CASE WHEN depression = 1 THEN 1 ELSE 0 END), 2) AS depression_rate_pct,

  SUM(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END) AS suicidal_yes,
  ROUND(100.0 * AVG(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END), 2) AS suicidal_yes_rate_pct
FROM {TABLE}
GROUP BY academic_pressure
ORDER BY academic_pressure;
"""
for r in fetch_all(sql):
    print(r)


{'level': 1, 'n': 4798, 'depressed': Decimal('932'), 'depression_rate_pct': Decimal('19.42'), 'suicidal_yes': Decimal('1975'), 'suicidal_yes_rate_pct': Decimal('41.16')}
{'level': 2, 'n': 4175, 'depressed': Decimal('1564'), 'depression_rate_pct': Decimal('37.46'), 'suicidal_yes': Decimal('2150'), 'suicidal_yes_rate_pct': Decimal('51.50')}
{'level': 3, 'n': 7447, 'depressed': Decimal('4476'), 'depression_rate_pct': Decimal('60.10'), 'suicidal_yes': Decimal('4877'), 'suicidal_yes_rate_pct': Decimal('65.49')}
{'level': 4, 'n': 5149, 'depressed': Decimal('3920'), 'depression_rate_pct': Decimal('76.13'), 'suicidal_yes': Decimal('3788'), 'suicidal_yes_rate_pct': Decimal('73.57')}
{'level': 5, 'n': 6284, 'depressed': Decimal('5408'), 'depression_rate_pct': Decimal('86.06'), 'suicidal_yes': Decimal('4831'), 'suicidal_yes_rate_pct': Decimal('76.88')}


##### **Q14. Depression rate by `financial_stress` (1–5).**


In [82]:
sql = f"""
SELECT
  financial_stress AS level,
  COUNT(*) AS n,

  SUM(CASE WHEN depression = 1 THEN 1 ELSE 0 END) AS depressed,
  ROUND(100.0 * AVG(CASE WHEN depression = 1 THEN 1 ELSE 0 END), 2) AS depression_rate_pct,

  SUM(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END) AS suicidal_yes,
  ROUND(100.0 * AVG(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END), 2) AS suicidal_yes_rate_pct
FROM {TABLE}
GROUP BY financial_stress
ORDER BY financial_stress;
"""
for r in fetch_all(sql):
    print(r)


{'level': 1, 'n': 5114, 'depressed': Decimal('1627'), 'depression_rate_pct': Decimal('31.81'), 'suicidal_yes': Decimal('2460'), 'suicidal_yes_rate_pct': Decimal('48.10')}
{'level': 2, 'n': 5057, 'depressed': Decimal('2172'), 'depression_rate_pct': Decimal('42.95'), 'suicidal_yes': Decimal('2740'), 'suicidal_yes_rate_pct': Decimal('54.18')}
{'level': 3, 'n': 5213, 'depressed': Decimal('3072'), 'depression_rate_pct': Decimal('58.93'), 'suicidal_yes': Decimal('3336'), 'suicidal_yes_rate_pct': Decimal('63.99')}
{'level': 4, 'n': 5769, 'depressed': Decimal('3985'), 'depression_rate_pct': Decimal('69.08'), 'suicidal_yes': Decimal('4012'), 'suicidal_yes_rate_pct': Decimal('69.54')}
{'level': 5, 'n': 6700, 'depressed': Decimal('5444'), 'depression_rate_pct': Decimal('81.25'), 'suicidal_yes': Decimal('5073'), 'suicidal_yes_rate_pct': Decimal('75.72')}


##### **Q15. Depression rate by `study_satisfaction` (1–5).**


In [83]:
sql = f"""
SELECT
  study_satisfaction AS level,
  COUNT(*) AS n,

  SUM(CASE WHEN depression = 1 THEN 1 ELSE 0 END) AS depressed,
  ROUND(100.0 * AVG(CASE WHEN depression = 1 THEN 1 ELSE 0 END), 2) AS depression_rate_pct,

  SUM(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END) AS suicidal_yes,
  ROUND(100.0 * AVG(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END), 2) AS suicidal_yes_rate_pct
FROM {TABLE}
GROUP BY study_satisfaction
ORDER BY study_satisfaction;
"""
for r in fetch_all(sql):
    print(r)


{'level': 1, 'n': 5444, 'depressed': Decimal('3851'), 'depression_rate_pct': Decimal('70.74'), 'suicidal_yes': Decimal('3740'), 'suicidal_yes_rate_pct': Decimal('68.70')}
{'level': 2, 'n': 5834, 'depressed': Decimal('3765'), 'depression_rate_pct': Decimal('64.54'), 'suicidal_yes': Decimal('3869'), 'suicidal_yes_rate_pct': Decimal('66.32')}
{'level': 3, 'n': 5810, 'depressed': Decimal('3344'), 'depression_rate_pct': Decimal('57.56'), 'suicidal_yes': Decimal('3710'), 'suicidal_yes_rate_pct': Decimal('63.86')}
{'level': 4, 'n': 6349, 'depressed': Decimal('3256'), 'depression_rate_pct': Decimal('51.28'), 'suicidal_yes': Decimal('3760'), 'suicidal_yes_rate_pct': Decimal('59.22')}
{'level': 5, 'n': 4416, 'depressed': Decimal('2084'), 'depression_rate_pct': Decimal('47.19'), 'suicidal_yes': Decimal('2542'), 'suicidal_yes_rate_pct': Decimal('57.56')}


##### **Q16. Depression rate by `sleep_duration`.**


In [84]:
sql = f"""
SELECT
  LOWER(TRIM(sleep_duration)) AS sleep_duration,
  COUNT(*) AS n,

  SUM(CASE WHEN depression = 1 THEN 1 ELSE 0 END) AS depressed,
  ROUND(100.0 * AVG(CASE WHEN depression = 1 THEN 1 ELSE 0 END), 2) AS depression_rate_pct,

  SUM(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END) AS suicidal_yes,
  ROUND(100.0 * AVG(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END), 2) AS suicidal_yes_rate_pct
FROM {TABLE}
GROUP BY LOWER(TRIM(sleep_duration))
ORDER BY n DESC, sleep_duration ASC;
"""
for r in fetch_all(sql):
    print(r)


{'sleep_duration': 'less than 5 hours', 'n': 8294, 'depressed': Decimal('5349'), 'depression_rate_pct': Decimal('64.49'), 'suicidal_yes': Decimal('5532'), 'suicidal_yes_rate_pct': Decimal('66.70')}
{'sleep_duration': '7-8 hours', 'n': 7335, 'depressed': Decimal('4361'), 'depression_rate_pct': Decimal('59.45'), 'suicidal_yes': Decimal('4762'), 'suicidal_yes_rate_pct': Decimal('64.92')}
{'sleep_duration': '5-6 hours', 'n': 6173, 'depressed': Decimal('3512'), 'depression_rate_pct': Decimal('56.89'), 'suicidal_yes': Decimal('3830'), 'suicidal_yes_rate_pct': Decimal('62.04')}
{'sleep_duration': 'more than 8 hours', 'n': 6033, 'depressed': Decimal('3069'), 'depression_rate_pct': Decimal('50.87'), 'suicidal_yes': Decimal('3486'), 'suicidal_yes_rate_pct': Decimal('57.78')}
{'sleep_duration': 'others', 'n': 18, 'depressed': Decimal('9'), 'depression_rate_pct': Decimal('50.00'), 'suicidal_yes': Decimal('11'), 'suicidal_yes_rate_pct': Decimal('61.11')}


##### **Q17. Depression rate by `dietary_habits`.**


In [85]:
sql = f"""
SELECT
  LOWER(TRIM(dietary_habits)) AS dietary_habits,
  COUNT(*) AS n,

  SUM(CASE WHEN depression = 1 THEN 1 ELSE 0 END) AS depressed,
  ROUND(100.0 * AVG(CASE WHEN depression = 1 THEN 1 ELSE 0 END), 2) AS depression_rate_pct,

  SUM(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END) AS suicidal_yes,
  ROUND(100.0 * AVG(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END), 2) AS suicidal_yes_rate_pct
FROM {TABLE}
GROUP BY LOWER(TRIM(dietary_habits))
ORDER BY n DESC, dietary_habits ASC;
"""
for r in fetch_all(sql):
    print(r)


{'dietary_habits': 'unhealthy', 'n': 10303, 'depressed': Decimal('7287'), 'depression_rate_pct': Decimal('70.73'), 'suicidal_yes': Decimal('7175'), 'suicidal_yes_rate_pct': Decimal('69.64')}
{'dietary_habits': 'moderate', 'n': 9903, 'depressed': Decimal('5543'), 'depression_rate_pct': Decimal('55.97'), 'suicidal_yes': Decimal('6156'), 'suicidal_yes_rate_pct': Decimal('62.16')}
{'dietary_habits': 'healthy', 'n': 7635, 'depressed': Decimal('3462'), 'depression_rate_pct': Decimal('45.34'), 'suicidal_yes': Decimal('4280'), 'suicidal_yes_rate_pct': Decimal('56.06')}
{'dietary_habits': 'others', 'n': 12, 'depressed': Decimal('8'), 'depression_rate_pct': Decimal('66.67'), 'suicidal_yes': Decimal('10'), 'suicidal_yes_rate_pct': Decimal('83.33')}


##### **Q18. Depression rate by `work_study_hours` bins (0–3, 4–6, 7–9, 10–12).**


In [86]:
sql = f"""
SELECT
  CASE
    WHEN work_study_hours BETWEEN 0 AND 3 THEN '0-3'
    WHEN work_study_hours BETWEEN 4 AND 6 THEN '4-6'
    WHEN work_study_hours BETWEEN 7 AND 9 THEN '7-9'
    WHEN work_study_hours BETWEEN 10 AND 12 THEN '10-12'
    ELSE 'unknown'
  END AS hours_bin,
  COUNT(*) AS n,

  SUM(CASE WHEN depression = 1 THEN 1 ELSE 0 END) AS depressed,
  ROUND(100.0 * AVG(CASE WHEN depression = 1 THEN 1 ELSE 0 END), 2) AS depression_rate_pct,

  SUM(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END) AS suicidal_yes,
  ROUND(100.0 * AVG(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END), 2) AS suicidal_yes_rate_pct
FROM {TABLE}
GROUP BY
  CASE
    WHEN work_study_hours BETWEEN 0 AND 3 THEN '0-3'
    WHEN work_study_hours BETWEEN 4 AND 6 THEN '4-6'
    WHEN work_study_hours BETWEEN 7 AND 9 THEN '7-9'
    WHEN work_study_hours BETWEEN 10 AND 12 THEN '10-12'
    ELSE 'unknown'
  END
ORDER BY FIELD(hours_bin, '0-3','4-6','7-9','10-12','unknown');
"""
for r in fetch_all(sql):
    print(r)


{'hours_bin': '0-3', 'n': 5895, 'depressed': Decimal('2450'), 'depression_rate_pct': Decimal('41.56'), 'suicidal_yes': Decimal('3151'), 'suicidal_yes_rate_pct': Decimal('53.45')}
{'hours_bin': '4-6', 'n': 5147, 'depressed': Decimal('2802'), 'depression_rate_pct': Decimal('54.44'), 'suicidal_yes': Decimal('3155'), 'suicidal_yes_rate_pct': Decimal('61.30')}
{'hours_bin': '7-9', 'n': 6530, 'depressed': Decimal('3953'), 'depression_rate_pct': Decimal('60.54'), 'suicidal_yes': Decimal('4157'), 'suicidal_yes_rate_pct': Decimal('63.66')}
{'hours_bin': '10-12', 'n': 10281, 'depressed': Decimal('7095'), 'depression_rate_pct': Decimal('69.01'), 'suicidal_yes': Decimal('7158'), 'suicidal_yes_rate_pct': Decimal('69.62')}


##### **Q19. Depression rate by `cgpa` bands (e.g., quartiles or <6.5, 6.5–7.5, 7.5–8.5, >8.5), and repeat within `degree_category`.**


In [87]:
# Depression rate by CGPA bands (<6.5, 6.5–7.5, 7.5–8.5, >8.5)
sql = f"""
SELECT
  CASE
    WHEN cgpa < 6.5 THEN '<6.5'
    WHEN cgpa >= 6.5 AND cgpa < 7.5 THEN '6.5-7.5'
    WHEN cgpa >= 7.5 AND cgpa < 8.5 THEN '7.5-8.5'
    WHEN cgpa >= 8.5 THEN '>8.5'
    ELSE 'unknown'
  END AS cgpa_band,
  COUNT(*) AS n,

  SUM(CASE WHEN depression = 1 THEN 1 ELSE 0 END) AS depressed,
  ROUND(100.0 * AVG(CASE WHEN depression = 1 THEN 1 ELSE 0 END), 2) AS depression_rate_pct,

  SUM(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END) AS suicidal_yes,
  ROUND(100.0 * AVG(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END), 2) AS suicidal_yes_rate_pct
FROM {TABLE}
GROUP BY
  CASE
    WHEN cgpa < 6.5 THEN '<6.5'
    WHEN cgpa >= 6.5 AND cgpa < 7.5 THEN '6.5-7.5'
    WHEN cgpa >= 7.5 AND cgpa < 8.5 THEN '7.5-8.5'
    WHEN cgpa >= 8.5 THEN '>8.5'
    ELSE 'unknown'
  END
ORDER BY FIELD(cgpa_band, '<6.5','6.5-7.5','7.5-8.5','>8.5','unknown');
"""
for r in fetch_all(sql):
    print(r)


{'cgpa_band': '<6.5', 'n': 7772, 'depressed': Decimal('4424'), 'depression_rate_pct': Decimal('56.92'), 'suicidal_yes': Decimal('4841'), 'suicidal_yes_rate_pct': Decimal('62.29')}
{'cgpa_band': '6.5-7.5', 'n': 4871, 'depressed': Decimal('2773'), 'depression_rate_pct': Decimal('56.93'), 'suicidal_yes': Decimal('3077'), 'suicidal_yes_rate_pct': Decimal('63.17')}
{'cgpa_band': '7.5-8.5', 'n': 5458, 'depressed': Decimal('3279'), 'depression_rate_pct': Decimal('60.08'), 'suicidal_yes': Decimal('3503'), 'suicidal_yes_rate_pct': Decimal('64.18')}
{'cgpa_band': '>8.5', 'n': 9752, 'depressed': Decimal('5824'), 'depression_rate_pct': Decimal('59.72'), 'suicidal_yes': Decimal('6200'), 'suicidal_yes_rate_pct': Decimal('63.58')}


In [88]:
# Repeat CGPA bands within degree_category
sql = f"""
SELECT
  LOWER(TRIM(degree_category)) AS degree_category,
  CASE
    WHEN cgpa < 6.5 THEN '<6.5'
    WHEN cgpa >= 6.5 AND cgpa < 7.5 THEN '6.5-7.5'
    WHEN cgpa >= 7.5 AND cgpa < 8.5 THEN '7.5-8.5'
    WHEN cgpa >= 8.5 THEN '>8.5'
    ELSE 'unknown'
  END AS cgpa_band,
  COUNT(*) AS n,

  SUM(CASE WHEN depression = 1 THEN 1 ELSE 0 END) AS depressed,
  ROUND(100.0 * AVG(CASE WHEN depression = 1 THEN 1 ELSE 0 END), 2) AS depression_rate_pct
FROM {TABLE}
GROUP BY
  LOWER(TRIM(degree_category)),
  CASE
    WHEN cgpa < 6.5 THEN '<6.5'
    WHEN cgpa >= 6.5 AND cgpa < 7.5 THEN '6.5-7.5'
    WHEN cgpa >= 7.5 AND cgpa < 8.5 THEN '7.5-8.5'
    WHEN cgpa >= 8.5 THEN '>8.5'
    ELSE 'unknown'
  END
ORDER BY
  degree_category,
  FIELD(cgpa_band, '<6.5','6.5-7.5','7.5-8.5','>8.5','unknown');
"""
for r in fetch_all(sql):
    print(r)


{'degree_category': 'others', 'cgpa_band': '<6.5', 'n': 11, 'depressed': Decimal('5'), 'depression_rate_pct': Decimal('45.45')}
{'degree_category': 'others', 'cgpa_band': '6.5-7.5', 'n': 6, 'depressed': Decimal('4'), 'depression_rate_pct': Decimal('66.67')}
{'degree_category': 'others', 'cgpa_band': '7.5-8.5', 'n': 7, 'depressed': Decimal('5'), 'depression_rate_pct': Decimal('71.43')}
{'degree_category': 'others', 'cgpa_band': '>8.5', 'n': 11, 'depressed': Decimal('7'), 'depression_rate_pct': Decimal('63.64')}
{'degree_category': 'pg', 'cgpa_band': '<6.5', 'n': 2250, 'depressed': Decimal('1157'), 'depression_rate_pct': Decimal('51.42')}
{'degree_category': 'pg', 'cgpa_band': '6.5-7.5', 'n': 1385, 'depressed': Decimal('719'), 'depression_rate_pct': Decimal('51.91')}
{'degree_category': 'pg', 'cgpa_band': '7.5-8.5', 'n': 1492, 'depressed': Decimal('799'), 'depression_rate_pct': Decimal('53.55')}
{'degree_category': 'pg', 'cgpa_band': '>8.5', 'n': 2786, 'depressed': Decimal('1536'), 'depr

##### **Q20. Depression lift for `family_history_mental_illness` (yes vs no).**


In [90]:
sql = f"""
SELECT
  LOWER(TRIM(family_history_mental_illness)) AS family_history,
  COUNT(*) AS n,

  SUM(CASE WHEN depression = 1 THEN 1 ELSE 0 END) AS depressed,
  ROUND(100.0 * AVG(CASE WHEN depression = 1 THEN 1 ELSE 0 END), 2) AS depression_rate_pct,

  SUM(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END) AS suicidal_yes,
  ROUND(100.0 * AVG(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END), 2) AS suicidal_yes_rate_pct
FROM {TABLE}
GROUP BY LOWER(TRIM(family_history_mental_illness))
ORDER BY n DESC, family_history ASC;
"""
for r in fetch_all(sql):
    print(r)


{'family_history': 'no', 'n': 14375, 'depressed': Decimal('8045'), 'depression_rate_pct': Decimal('55.97'), 'suicidal_yes': Decimal('8919'), 'suicidal_yes_rate_pct': Decimal('62.05')}
{'family_history': 'yes', 'n': 13478, 'depressed': Decimal('8255'), 'depression_rate_pct': Decimal('61.25'), 'suicidal_yes': Decimal('8702'), 'suicidal_yes_rate_pct': Decimal('64.56')}


### **High-Risk Combinations (Multivariate, Logical)**

##### **Q21. Depression rate by `academic_pressure × financial_stress` (5×5), sorted by highest risk AND highest count.**

In [ ]:
# Stress stacking
sql = f"""
SELECT
  academic_pressure,
  financial_stress,
  COUNT(*) AS n,

  SUM(CASE WHEN depression = 1 THEN 1 ELSE 0 END) AS depressed,
  ROUND(100.0 * AVG(CASE WHEN depression = 1 THEN 1 ELSE 0 END), 2) AS depression_rate_pct,

  SUM(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END) AS suicidal_yes,
  ROUND(100.0 * AVG(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END), 2) AS suicidal_yes_rate_pct
FROM {TABLE}
GROUP BY academic_pressure, financial_stress
ORDER BY depression_rate_pct DESC, n DESC, academic_pressure DESC, financial_stress DESC;
"""
for r in fetch_all(sql):
    print(r)


{'academic_pressure': 5, 'financial_stress': 5, 'n': 1915, 'depressed': Decimal('1827'), 'depression_rate_pct': Decimal('95.40'), 'suicidal_yes': Decimal('1581'), 'suicidal_yes_rate_pct': Decimal('82.56')}
{'academic_pressure': 4, 'financial_stress': 5, 'n': 1319, 'depressed': Decimal('1200'), 'depression_rate_pct': Decimal('90.98'), 'suicidal_yes': Decimal('1053'), 'suicidal_yes_rate_pct': Decimal('79.83')}
{'academic_pressure': 5, 'financial_stress': 4, 'n': 1334, 'depressed': Decimal('1202'), 'depression_rate_pct': Decimal('90.10'), 'suicidal_yes': Decimal('1054'), 'suicidal_yes_rate_pct': Decimal('79.01')}
{'academic_pressure': 5, 'financial_stress': 3, 'n': 1235, 'depressed': Decimal('1068'), 'depression_rate_pct': Decimal('86.48'), 'suicidal_yes': Decimal('957'), 'suicidal_yes_rate_pct': Decimal('77.49')}
{'academic_pressure': 4, 'financial_stress': 4, 'n': 1179, 'depressed': Decimal('983'), 'depression_rate_pct': Decimal('83.38'), 'suicidal_yes': Decimal('912'), 'suicidal_yes_ra

##### **Q22. Within high pressure (academic_pressure 4–5), does high satisfaction (4–5) reduce depression vs low satisfaction (1–2)?**


In [93]:
# Buffers inside risk
sql = f"""
SELECT
  CASE
    WHEN study_satisfaction IN (4,5) THEN 'high_satisfaction_4_5'
    WHEN study_satisfaction IN (1,2) THEN 'low_satisfaction_1_2'
    ELSE 'mid_satisfaction_3'
  END AS satisfaction_group,
  COUNT(*) AS n,

  SUM(CASE WHEN depression = 1 THEN 1 ELSE 0 END) AS depressed,
  ROUND(100.0 * AVG(CASE WHEN depression = 1 THEN 1 ELSE 0 END), 2) AS depression_rate_pct,

  SUM(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END) AS suicidal_yes,
  ROUND(100.0 * AVG(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END), 2) AS suicidal_yes_rate_pct
FROM {TABLE}
WHERE academic_pressure IN (4,5)
GROUP BY
  CASE
    WHEN study_satisfaction IN (4,5) THEN 'high_satisfaction_4_5'
    WHEN study_satisfaction IN (1,2) THEN 'low_satisfaction_1_2'
    ELSE 'mid_satisfaction_3'
  END
ORDER BY FIELD(satisfaction_group, 'low_satisfaction_1_2','mid_satisfaction_3','high_satisfaction_4_5');
"""
for r in fetch_all(sql):
    print(r)


{'satisfaction_group': 'low_satisfaction_1_2', 'n': 5462, 'depressed': Decimal('4721'), 'depression_rate_pct': Decimal('86.43'), 'suicidal_yes': Decimal('4221'), 'suicidal_yes_rate_pct': Decimal('77.28')}
{'satisfaction_group': 'mid_satisfaction_3', 'n': 2237, 'depressed': Decimal('1841'), 'depression_rate_pct': Decimal('82.30'), 'suicidal_yes': Decimal('1741'), 'suicidal_yes_rate_pct': Decimal('77.83')}
{'satisfaction_group': 'high_satisfaction_4_5', 'n': 3734, 'depressed': Decimal('2766'), 'depression_rate_pct': Decimal('74.08'), 'suicidal_yes': Decimal('2657'), 'suicidal_yes_rate_pct': Decimal('71.16')}


##### **Q23. Depression rate by `work_study_hours bin × sleep_duration`, and identify the highest-risk AND largest “burnout zone”.**

In [ ]:
# Burnout zone
# Depression rate by work_study_hours bin × sleep_duration
sql = f"""
SELECT
  CASE
    WHEN work_study_hours BETWEEN 0 AND 3 THEN '0-3'
    WHEN work_study_hours BETWEEN 4 AND 6 THEN '4-6'
    WHEN work_study_hours BETWEEN 7 AND 9 THEN '7-9'
    WHEN work_study_hours BETWEEN 10 AND 12 THEN '10-12'
    ELSE 'unknown'
  END AS hours_bin,
  LOWER(TRIM(sleep_duration)) AS sleep_duration,
  COUNT(*) AS n,

  SUM(CASE WHEN depression = 1 THEN 1 ELSE 0 END) AS depressed,
  ROUND(100.0 * AVG(CASE WHEN depression = 1 THEN 1 ELSE 0 END), 2) AS depression_rate_pct,

  SUM(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END) AS suicidal_yes,
  ROUND(100.0 * AVG(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END), 2) AS suicidal_yes_rate_pct
FROM {TABLE}
GROUP BY
  CASE
    WHEN work_study_hours BETWEEN 0 AND 3 THEN '0-3'
    WHEN work_study_hours BETWEEN 4 AND 6 THEN '4-6'
    WHEN work_study_hours BETWEEN 7 AND 9 THEN '7-9'
    WHEN work_study_hours BETWEEN 10 AND 12 THEN '10-12'
    ELSE 'unknown'
  END,
  LOWER(TRIM(sleep_duration))
ORDER BY
  FIELD(hours_bin, '0-3','4-6','7-9','10-12','unknown'),
  n DESC,
  sleep_duration ASC;
"""
for r in fetch_all(sql):
    print(r)


{'hours_bin': '0-3', 'sleep_duration': 'less than 5 hours', 'n': 1734, 'depressed': Decimal('860'), 'depression_rate_pct': Decimal('49.60'), 'suicidal_yes': Decimal('1002'), 'suicidal_yes_rate_pct': Decimal('57.79')}
{'hours_bin': '0-3', 'sleep_duration': '7-8 hours', 'n': 1476, 'depressed': Decimal('595'), 'depression_rate_pct': Decimal('40.31'), 'suicidal_yes': Decimal('787'), 'suicidal_yes_rate_pct': Decimal('53.32')}
{'hours_bin': '0-3', 'sleep_duration': 'more than 8 hours', 'n': 1376, 'depressed': Decimal('472'), 'depression_rate_pct': Decimal('34.30'), 'suicidal_yes': Decimal('657'), 'suicidal_yes_rate_pct': Decimal('47.75')}
{'hours_bin': '0-3', 'sleep_duration': '5-6 hours', 'n': 1303, 'depressed': Decimal('521'), 'depression_rate_pct': Decimal('39.98'), 'suicidal_yes': Decimal('701'), 'suicidal_yes_rate_pct': Decimal('53.80')}
{'hours_bin': '0-3', 'sleep_duration': 'others', 'n': 6, 'depressed': Decimal('2'), 'depression_rate_pct': Decimal('33.33'), 'suicidal_yes': Decimal('4

In [11]:
# Identify the “burnout zone” = highest risk AND meaningful volume
MIN_N = 200

sql = f"""
SELECT
  hours_bin,
  sleep_duration,
  n,
  depressed,
  depression_rate_pct,
  suicidal_yes,
  suicidal_yes_rate_pct
FROM (
  SELECT
    CASE
      WHEN work_study_hours BETWEEN 0 AND 3 THEN '0-3'
      WHEN work_study_hours BETWEEN 4 AND 6 THEN '4-6'
      WHEN work_study_hours BETWEEN 7 AND 9 THEN '7-9'
      WHEN work_study_hours BETWEEN 10 AND 12 THEN '10-12'
      ELSE 'unknown'
    END AS hours_bin,
    LOWER(TRIM(sleep_duration)) AS sleep_duration,
    COUNT(*) AS n,
    SUM(CASE WHEN depression = 1 THEN 1 ELSE 0 END) AS depressed,
    ROUND(100.0 * AVG(CASE WHEN depression = 1 THEN 1 ELSE 0 END), 2) AS depression_rate_pct,
    SUM(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END) AS suicidal_yes,
    ROUND(100.0 * AVG(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END), 2) AS suicidal_yes_rate_pct
  FROM {TABLE}
  GROUP BY
    CASE
      WHEN work_study_hours BETWEEN 0 AND 3 THEN '0-3'
      WHEN work_study_hours BETWEEN 4 AND 6 THEN '4-6'
      WHEN work_study_hours BETWEEN 7 AND 9 THEN '7-9'
      WHEN work_study_hours BETWEEN 10 AND 12 THEN '10-12'
      ELSE 'unknown'
    END,
    LOWER(TRIM(sleep_duration))
) t
WHERE n >= {MIN_N}
ORDER BY depression_rate_pct DESC, n DESC
LIMIT 10;
"""
for r in fetch_all(sql):
    print(r)


{'hours_bin': '10-12', 'sleep_duration': 'less than 5 hours', 'n': 3065, 'depressed': Decimal('2263'), 'depression_rate_pct': Decimal('73.83'), 'suicidal_yes': Decimal('2234'), 'suicidal_yes_rate_pct': Decimal('72.89')}
{'hours_bin': '10-12', 'sleep_duration': '7-8 hours', 'n': 2810, 'depressed': Decimal('1955'), 'depression_rate_pct': Decimal('69.57'), 'suicidal_yes': Decimal('1986'), 'suicidal_yes_rate_pct': Decimal('70.68')}
{'hours_bin': '10-12', 'sleep_duration': '5-6 hours', 'n': 2423, 'depressed': Decimal('1638'), 'depression_rate_pct': Decimal('67.60'), 'suicidal_yes': Decimal('1656'), 'suicidal_yes_rate_pct': Decimal('68.35')}
{'hours_bin': '7-9', 'sleep_duration': 'less than 5 hours', 'n': 1986, 'depressed': Decimal('1318'), 'depression_rate_pct': Decimal('66.36'), 'suicidal_yes': Decimal('1319'), 'suicidal_yes_rate_pct': Decimal('66.41')}
{'hours_bin': '10-12', 'sleep_duration': 'more than 8 hours', 'n': 1976, 'depressed': Decimal('1235'), 'depression_rate_pct': Decimal('62.

##### **Q24. Depression rate by `sleep_duration × dietary_habits`, and which combo contributes most depressed students by volume?**



In [97]:
# Depression rate by sleep_duration × dietary_habits
sql = f"""
SELECT
  LOWER(TRIM(sleep_duration)) AS sleep_duration,
  LOWER(TRIM(dietary_habits)) AS dietary_habits,
  COUNT(*) AS n,

  SUM(CASE WHEN depression = 1 THEN 1 ELSE 0 END) AS depressed,
  ROUND(100.0 * AVG(CASE WHEN depression = 1 THEN 1 ELSE 0 END), 2) AS depression_rate_pct,

  SUM(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END) AS suicidal_yes,
  ROUND(100.0 * AVG(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END), 2) AS suicidal_yes_rate_pct
FROM {TABLE}
GROUP BY LOWER(TRIM(sleep_duration)), LOWER(TRIM(dietary_habits))
ORDER BY depression_rate_pct DESC, n DESC, sleep_duration ASC, dietary_habits ASC;
"""
for r in fetch_all(sql):
    print(r)


{'sleep_duration': '5-6 hours', 'dietary_habits': 'others', 'n': 3, 'depressed': Decimal('3'), 'depression_rate_pct': Decimal('100.00'), 'suicidal_yes': Decimal('2'), 'suicidal_yes_rate_pct': Decimal('66.67')}
{'sleep_duration': 'more than 8 hours', 'dietary_habits': 'others', 'n': 1, 'depressed': Decimal('1'), 'depression_rate_pct': Decimal('100.00'), 'suicidal_yes': Decimal('1'), 'suicidal_yes_rate_pct': Decimal('100.00')}
{'sleep_duration': 'less than 5 hours', 'dietary_habits': 'unhealthy', 'n': 3061, 'depressed': Decimal('2329'), 'depression_rate_pct': Decimal('76.09'), 'suicidal_yes': Decimal('2223'), 'suicidal_yes_rate_pct': Decimal('72.62')}
{'sleep_duration': '7-8 hours', 'dietary_habits': 'unhealthy', 'n': 2817, 'depressed': Decimal('2024'), 'depression_rate_pct': Decimal('71.85'), 'suicidal_yes': Decimal('1996'), 'suicidal_yes_rate_pct': Decimal('70.86')}
{'sleep_duration': '5-6 hours', 'dietary_habits': 'unhealthy', 'n': 2179, 'depressed': Decimal('1483'), 'depression_rate_

In [98]:
# Which combo contributes most depressed students by volume?
MIN_N = 200

sql = f"""
SELECT
  sleep_duration,
  dietary_habits,
  n,
  depressed,
  depression_rate_pct,
  suicidal_yes,
  suicidal_yes_rate_pct
FROM (
  SELECT
    LOWER(TRIM(sleep_duration)) AS sleep_duration,
    LOWER(TRIM(dietary_habits)) AS dietary_habits,
    COUNT(*) AS n,
    SUM(CASE WHEN depression = 1 THEN 1 ELSE 0 END) AS depressed,
    ROUND(100.0 * AVG(CASE WHEN depression = 1 THEN 1 ELSE 0 END), 2) AS depression_rate_pct,
    SUM(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END) AS suicidal_yes,
    ROUND(100.0 * AVG(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END), 2) AS suicidal_yes_rate_pct
  FROM {TABLE}
  GROUP BY LOWER(TRIM(sleep_duration)), LOWER(TRIM(dietary_habits))
) t
WHERE n >= {MIN_N}
ORDER BY depressed DESC, depression_rate_pct DESC, n DESC
LIMIT 10;
"""
for r in fetch_all(sql):
    print(r)


{'sleep_duration': 'less than 5 hours', 'dietary_habits': 'unhealthy', 'n': 3061, 'depressed': Decimal('2329'), 'depression_rate_pct': Decimal('76.09'), 'suicidal_yes': Decimal('2223'), 'suicidal_yes_rate_pct': Decimal('72.62')}
{'sleep_duration': '7-8 hours', 'dietary_habits': 'unhealthy', 'n': 2817, 'depressed': Decimal('2024'), 'depression_rate_pct': Decimal('71.85'), 'suicidal_yes': Decimal('1996'), 'suicidal_yes_rate_pct': Decimal('70.86')}
{'sleep_duration': 'less than 5 hours', 'dietary_habits': 'moderate', 'n': 3034, 'depressed': Decimal('1917'), 'depression_rate_pct': Decimal('63.18'), 'suicidal_yes': Decimal('2011'), 'suicidal_yes_rate_pct': Decimal('66.28')}
{'sleep_duration': '5-6 hours', 'dietary_habits': 'unhealthy', 'n': 2179, 'depressed': Decimal('1483'), 'depression_rate_pct': Decimal('68.06'), 'suicidal_yes': Decimal('1486'), 'suicidal_yes_rate_pct': Decimal('68.20')}
{'sleep_duration': 'more than 8 hours', 'dietary_habits': 'unhealthy', 'n': 2238, 'depressed': Decima

##### **Q25. Within each `wellness` tier, what is suicidal_thoughts rate and depression rate?**


In [12]:

sql = f"""
SELECT
  LOWER(TRIM(wellness)) AS wellness,
  COUNT(*) AS n,

  SUM(CASE WHEN depression = 1 THEN 1 ELSE 0 END) AS depressed,
  ROUND(100.0 * AVG(CASE WHEN depression = 1 THEN 1 ELSE 0 END), 2) AS depression_rate_pct,

  SUM(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END) AS suicidal_yes,
  ROUND(100.0 * AVG(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END), 2) AS suicidal_yes_rate_pct
FROM {TABLE}
GROUP BY LOWER(TRIM(wellness))
ORDER BY n DESC, wellness ASC;
"""
for r in fetch_all(sql):
    print(r)


{'wellness': 'moderate', 'n': 12373, 'depressed': Decimal('9291'), 'depression_rate_pct': Decimal('75.09'), 'suicidal_yes': Decimal('9699'), 'suicidal_yes_rate_pct': Decimal('78.39')}
{'wellness': 'high', 'n': 11082, 'depressed': Decimal('2803'), 'depression_rate_pct': Decimal('25.29'), 'suicidal_yes': Decimal('3739'), 'suicidal_yes_rate_pct': Decimal('33.74')}
{'wellness': 'low', 'n': 4398, 'depressed': Decimal('4206'), 'depression_rate_pct': Decimal('95.63'), 'suicidal_yes': Decimal('4183'), 'suicidal_yes_rate_pct': Decimal('95.11')}


##### **Q26. What is risk + count for students with `academic_pressure≥4 AND financial_stress≥4 AND sleep<5 hours`?**


In [99]:
sql = f"""
SELECT
  COUNT(*) AS n,
  SUM(CASE WHEN depression = 1 THEN 1 ELSE 0 END) AS depressed,
  ROUND(100.0 * AVG(CASE WHEN depression = 1 THEN 1 ELSE 0 END), 2) AS depression_rate_pct,
  SUM(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END) AS suicidal_yes,
  ROUND(100.0 * AVG(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END), 2) AS suicidal_yes_rate_pct
FROM {TABLE}
WHERE academic_pressure >= 4
  AND financial_stress >= 4
  AND LOWER(TRIM(sleep_duration)) = 'less than 5 hours';
"""
print(fetch_one(sql))


{'n': 1870, 'depressed': Decimal('1729'), 'depression_rate_pct': Decimal('92.46'), 'suicidal_yes': Decimal('1547'), 'suicidal_yes_rate_pct': Decimal('82.73')}


##### **Q27. What is risk + count for students with `work_study_hours≥10 AND sleep<5 hours AND study_satisfaction≤2`?**

In [13]:
sql = f"""
SELECT
  COUNT(*) AS n,
  SUM(CASE WHEN depression = 1 THEN 1 ELSE 0 END) AS depressed,
  ROUND(100.0 * AVG(CASE WHEN depression = 1 THEN 1 ELSE 0 END), 2) AS depression_rate_pct,
  SUM(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END) AS suicidal_yes,
  ROUND(100.0 * AVG(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END), 2) AS suicidal_yes_rate_pct
FROM {TABLE}
WHERE work_study_hours >= 10
  AND LOWER(TRIM(sleep_duration)) = 'less than 5 hours'
  AND study_satisfaction <= 2;
"""
print(fetch_one(sql))


{'n': 1320, 'depressed': Decimal('1046'), 'depression_rate_pct': Decimal('79.24'), 'suicidal_yes': Decimal('987'), 'suicidal_yes_rate_pct': Decimal('74.77')}


**Q28. Preventive Watchlist: how many students are `depression=0 AND red_flag≥5`, and what are their top driver profiles?**

In [101]:
# Preventive Watchlist size: depression=0 AND red_flag≥5
sql = f"""
SELECT
  COUNT(*) AS preventive_watchlist_n,
  SUM(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END) AS suicidal_yes,
  ROUND(100.0 * AVG(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END), 2) AS suicidal_yes_rate_pct
FROM {TABLE}
WHERE depression = 0
  AND red_flag >= 5;
"""
print(fetch_one(sql))


{'preventive_watchlist_n': 192, 'suicidal_yes': Decimal('149'), 'suicidal_yes_rate_pct': Decimal('77.60')}


In [102]:
# Top driver profiles inside the preventive watchlist (most actionable patterns)
sql = f"""
SELECT
  academic_pressure,
  financial_stress,
  study_satisfaction,
  LOWER(TRIM(sleep_duration)) AS sleep_duration,
  LOWER(TRIM(dietary_habits)) AS dietary_habits,
  CASE WHEN work_study_hours >= 10 THEN 'hours_10_12' ELSE 'hours_0_9' END AS hours_group,
  COUNT(*) AS n,
  SUM(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END) AS suicidal_yes,
  ROUND(100.0 * AVG(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END), 2) AS suicidal_yes_rate_pct
FROM {TABLE}
WHERE depression = 0
  AND red_flag >= 5
GROUP BY
  academic_pressure, financial_stress, study_satisfaction,
  LOWER(TRIM(sleep_duration)), LOWER(TRIM(dietary_habits)),
  CASE WHEN work_study_hours >= 10 THEN 'hours_10_12' ELSE 'hours_0_9' END
ORDER BY n DESC
LIMIT 15;
"""
for r in fetch_all(sql):
    print(r)


{'academic_pressure': 5, 'financial_stress': 1, 'study_satisfaction': 2, 'sleep_duration': 'less than 5 hours', 'dietary_habits': 'healthy', 'hours_group': 'hours_10_12', 'n': 4, 'suicidal_yes': Decimal('4'), 'suicidal_yes_rate_pct': Decimal('100.00')}
{'academic_pressure': 3, 'financial_stress': 4, 'study_satisfaction': 2, 'sleep_duration': 'less than 5 hours', 'dietary_habits': 'unhealthy', 'hours_group': 'hours_10_12', 'n': 4, 'suicidal_yes': Decimal('0'), 'suicidal_yes_rate_pct': Decimal('0.00')}
{'academic_pressure': 4, 'financial_stress': 4, 'study_satisfaction': 1, 'sleep_duration': '5-6 hours', 'dietary_habits': 'healthy', 'hours_group': 'hours_10_12', 'n': 3, 'suicidal_yes': Decimal('3'), 'suicidal_yes_rate_pct': Decimal('100.00')}
{'academic_pressure': 5, 'financial_stress': 5, 'study_satisfaction': 2, 'sleep_duration': 'less than 5 hours', 'dietary_habits': 'healthy', 'hours_group': 'hours_0_9', 'n': 3, 'suicidal_yes': Decimal('3'), 'suicidal_yes_rate_pct': Decimal('100.00')

##### **Q29. Hidden Risk: how many are `depression=1 AND red_flag≤2`, and what patterns dominate them?**


In [103]:
# Hidden Risk size: depression=1 AND red_flag≤2
sql = f"""
SELECT
  COUNT(*) AS hidden_risk_n,
  SUM(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END) AS suicidal_yes,
  ROUND(100.0 * AVG(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END), 2) AS suicidal_yes_rate_pct
FROM {TABLE}
WHERE depression = 1
  AND red_flag <= 2;
"""
print(fetch_one(sql))


{'hidden_risk_n': 2803, 'suicidal_yes': Decimal('1882'), 'suicidal_yes_rate_pct': Decimal('67.14')}


In [104]:
# What patterns dominate hidden risk? (top profiles by volume)
sql = f"""
SELECT
  academic_pressure,
  financial_stress,
  study_satisfaction,
  LOWER(TRIM(sleep_duration)) AS sleep_duration,
  LOWER(TRIM(dietary_habits)) AS dietary_habits,
  CASE WHEN work_study_hours >= 10 THEN 'hours_10_12' ELSE 'hours_0_9' END AS hours_group,
  LOWER(TRIM(family_history_mental_illness)) AS family_history,
  COUNT(*) AS n,
  ROUND(100.0 * AVG(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END), 2) AS suicidal_yes_rate_pct
FROM {TABLE}
WHERE depression = 1
  AND red_flag <= 2
GROUP BY
  academic_pressure, financial_stress, study_satisfaction,
  LOWER(TRIM(sleep_duration)), LOWER(TRIM(dietary_habits)),
  CASE WHEN work_study_hours >= 10 THEN 'hours_10_12' ELSE 'hours_0_9' END,
  LOWER(TRIM(family_history_mental_illness))
ORDER BY n DESC
LIMIT 15;
"""
for r in fetch_all(sql):
    print(r)


{'academic_pressure': 3, 'financial_stress': 4, 'study_satisfaction': 5, 'sleep_duration': '7-8 hours', 'dietary_habits': 'moderate', 'hours_group': 'hours_0_9', 'family_history': 'no', 'n': 11, 'suicidal_yes_rate_pct': Decimal('100.00')}
{'academic_pressure': 3, 'financial_stress': 3, 'study_satisfaction': 3, 'sleep_duration': '7-8 hours', 'dietary_habits': 'moderate', 'hours_group': 'hours_0_9', 'family_history': 'yes', 'n': 10, 'suicidal_yes_rate_pct': Decimal('100.00')}
{'academic_pressure': 3, 'financial_stress': 4, 'study_satisfaction': 3, 'sleep_duration': '7-8 hours', 'dietary_habits': 'moderate', 'hours_group': 'hours_0_9', 'family_history': 'no', 'n': 10, 'suicidal_yes_rate_pct': Decimal('70.00')}
{'academic_pressure': 3, 'financial_stress': 1, 'study_satisfaction': 4, 'sleep_duration': '7-8 hours', 'dietary_habits': 'moderate', 'hours_group': 'hours_0_9', 'family_history': 'yes', 'n': 10, 'suicidal_yes_rate_pct': Decimal('90.00')}
{'academic_pressure': 3, 'financial_stress':

##### **Q30. Top 5 cities by depressed student volume (count), with depression% and suicidal% shown together.**


In [105]:
MIN_N = 300

sql = f"""
SELECT
  LOWER(TRIM(city)) AS city,
  COUNT(*) AS n,
  SUM(CASE WHEN depression = 1 THEN 1 ELSE 0 END) AS depressed,
  ROUND(100.0 * AVG(CASE WHEN depression = 1 THEN 1 ELSE 0 END), 2) AS depression_rate_pct,
  ROUND(100.0 * AVG(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END), 2) AS suicidal_yes_rate_pct
FROM {TABLE}
GROUP BY LOWER(TRIM(city))
HAVING COUNT(*) >= {MIN_N}
ORDER BY depressed DESC, depression_rate_pct DESC, n DESC
LIMIT 5;
"""
for r in fetch_all(sql):
    print(r)


{'city': 'kalyan', 'n': 1566, 'depressed': Decimal('928'), 'depression_rate_pct': Decimal('59.26'), 'suicidal_yes_rate_pct': Decimal('64.05')}
{'city': 'hyderabad', 'n': 1338, 'depressed': Decimal('896'), 'depression_rate_pct': Decimal('66.97'), 'suicidal_yes_rate_pct': Decimal('67.86')}
{'city': 'srinagar', 'n': 1372, 'depressed': Decimal('763'), 'depression_rate_pct': Decimal('55.61'), 'suicidal_yes_rate_pct': Decimal('63.05')}
{'city': 'vasai-virar', 'n': 1289, 'depressed': Decimal('738'), 'depression_rate_pct': Decimal('57.25'), 'suicidal_yes_rate_pct': Decimal('61.68')}
{'city': 'thane', 'n': 1139, 'depressed': Decimal('673'), 'depression_rate_pct': Decimal('59.09'), 'suicidal_yes_rate_pct': Decimal('65.06')}


##### **Q31. Top 5 degree programs by depressed volume, with depression% and suicidal% together.**


In [106]:
MIN_N = 300

sql = f"""
SELECT
  LOWER(TRIM(degree)) AS degree,
  COUNT(*) AS n,
  SUM(CASE WHEN depression = 1 THEN 1 ELSE 0 END) AS depressed,
  ROUND(100.0 * AVG(CASE WHEN depression = 1 THEN 1 ELSE 0 END), 2) AS depression_rate_pct,
  ROUND(100.0 * AVG(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END), 2) AS suicidal_yes_rate_pct
FROM {TABLE}
GROUP BY LOWER(TRIM(degree))
HAVING COUNT(*) >= {MIN_N}
ORDER BY depressed DESC, depression_rate_pct DESC, n DESC
LIMIT 5;
"""
for r in fetch_all(sql):
    print(r)


{'degree': 'class 12', 'n': 6070, 'depressed': Decimal('4297'), 'depression_rate_pct': Decimal('70.79'), 'suicidal_yes_rate_pct': Decimal('69.03')}
{'degree': 'b.ed', 'n': 1863, 'depressed': Decimal('1018'), 'depression_rate_pct': Decimal('54.64'), 'suicidal_yes_rate_pct': Decimal('61.94')}
{'degree': 'b.arch', 'n': 1477, 'depressed': Decimal('870'), 'depression_rate_pct': Decimal('58.90'), 'suicidal_yes_rate_pct': Decimal('61.75')}
{'degree': 'b.com', 'n': 1505, 'depressed': Decimal('853'), 'depression_rate_pct': Decimal('56.68'), 'suicidal_yes_rate_pct': Decimal('63.19')}
{'degree': 'bca', 'n': 1431, 'depressed': Decimal('817'), 'depression_rate_pct': Decimal('57.09'), 'suicidal_yes_rate_pct': Decimal('63.17')}


### **Feature Enginnered EDA Validation**

##### **1) Does risk increase as red_flag increases? (multi-variant)**

In [ ]:
# red_flag ladder (0–7): N + depression% + suicidal%
sql = f"""
SELECT
  red_flag,
  COUNT(*) AS n,
  ROUND(100.0 * AVG(CASE WHEN depression = 1 THEN 1 ELSE 0 END), 2) AS depression_pct,
  ROUND(100.0 * AVG(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END), 2) AS suicidal_pct
FROM {TABLE}
GROUP BY red_flag
ORDER BY red_flag;
"""
for r in fetch_all(sql):
    print(r)


{'red_flag': 0, 'n': 1320, 'depression_pct': Decimal('3.41'), 'suicidal_pct': Decimal('0.00')}
{'red_flag': 1, 'n': 3962, 'depression_pct': Decimal('14.84'), 'suicidal_pct': Decimal('22.87')}
{'red_flag': 2, 'n': 5800, 'depression_pct': Decimal('37.41'), 'suicidal_pct': Decimal('48.84')}
{'red_flag': 3, 'n': 6680, 'depression_pct': Decimal('65.63'), 'suicidal_pct': Decimal('71.36')}
{'red_flag': 4, 'n': 5693, 'depression_pct': Decimal('86.19'), 'suicidal_pct': Decimal('86.63')}
{'red_flag': 5, 'n': 3233, 'depression_pct': Decimal('94.71'), 'suicidal_pct': Decimal('94.34')}
{'red_flag': 6, 'n': 1015, 'depression_pct': Decimal('98.03'), 'suicidal_pct': Decimal('96.85')}
{'red_flag': 7, 'n': 150, 'depression_pct': Decimal('99.33'), 'suicidal_pct': Decimal('100.00')}


##### **2) Do the 3 wellness tiers separate risk clearly? (bi + multi-variant)**

In [ ]:
# wellness tiers: N + depression% + suicidal%
sql = f"""
SELECT
  LOWER(TRIM(wellness)) AS wellness,
  COUNT(*) AS n,
  ROUND(100.0 * AVG(CASE WHEN depression = 1 THEN 1 ELSE 0 END), 2) AS depression_pct,
  ROUND(100.0 * AVG(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END), 2) AS suicidal_pct
FROM {TABLE}
GROUP BY LOWER(TRIM(wellness))
ORDER BY n DESC, wellness ASC;
"""
for r in fetch_all(sql):
    print(r)


{'wellness': 'moderate', 'n': 12373, 'depression_pct': Decimal('75.09'), 'suicidal_pct': Decimal('78.39')}
{'wellness': 'high', 'n': 11082, 'depression_pct': Decimal('25.29'), 'suicidal_pct': Decimal('33.74')}
{'wellness': 'low', 'n': 4398, 'depression_pct': Decimal('95.63'), 'suicidal_pct': Decimal('95.11')}


##### **3) Is the action queue (support_priority) operationally realistic? (multi-variant)**

In [7]:
# support_priority queue: N + %total + suicidal%
sql = f"""
SELECT
  LOWER(TRIM(support_priority)) AS support_priority,
  COUNT(*) AS n,
  ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM {TABLE}), 2) AS pct_of_total,
  ROUND(100.0 * AVG(CASE WHEN LOWER(TRIM(suicidal_thoughts)) = 'yes' THEN 1 ELSE 0 END), 2) AS suicidal_pct
FROM {TABLE}
GROUP BY LOWER(TRIM(support_priority))
ORDER BY n DESC, support_priority ASC;
"""
for r in fetch_all(sql):
    print(r)


{'support_priority': 'moderate priority', 'n': 12094, 'pct_of_total': Decimal('43.42'), 'suicidal_pct': Decimal('81.80')}
{'support_priority': 'stable', 'n': 8279, 'pct_of_total': Decimal('29.72'), 'suicidal_pct': Decimal('22.43')}
{'support_priority': 'preventive watchlist', 'n': 3082, 'pct_of_total': Decimal('11.07'), 'suicidal_pct': Decimal('54.77')}
{'support_priority': 'high priority', 'n': 3062, 'pct_of_total': Decimal('10.99'), 'suicidal_pct': Decimal('95.40')}
{'support_priority': 'critical', 'n': 1144, 'pct_of_total': Decimal('4.11'), 'suicidal_pct': Decimal('97.29')}
{'support_priority': 'preventive high risk', 'n': 192, 'pct_of_total': Decimal('0.69'), 'suicidal_pct': Decimal('77.60')}


##### **4) How big is the “urgent workload” vs “prevention workload”? (bi-variant)**

In [8]:
# Urgent vs Prevention workload: count + %total + suicidal%
sql = f"""
SELECT
  grp,
  n,
  ROUND(100.0 * n / total_n, 2) AS pct_of_total,
  ROUND(100.0 * suicidal_yes / n, 2) AS suicidal_pct
FROM (
  SELECT
    'urgent (critical + high priority)' AS grp,
    COUNT(*) AS n,
    SUM(CASE WHEN LOWER(TRIM(suicidal_thoughts))='yes' THEN 1 ELSE 0 END) AS suicidal_yes,
    (SELECT COUNT(*) FROM {TABLE}) AS total_n
  FROM {TABLE}
  WHERE LOWER(TRIM(support_priority)) IN ('critical','high priority')

  UNION ALL

  SELECT
    'prevention (preventive high risk + watchlist)' AS grp,
    COUNT(*) AS n,
    SUM(CASE WHEN LOWER(TRIM(suicidal_thoughts))='yes' THEN 1 ELSE 0 END) AS suicidal_yes,
    (SELECT COUNT(*) FROM {TABLE}) AS total_n
  FROM {TABLE}
  WHERE LOWER(TRIM(support_priority)) IN ('preventive high risk','preventive watchlist')
) t;
"""
for r in fetch_all(sql):
    print(r)


{'grp': 'urgent (critical + high priority)', 'n': 4206, 'pct_of_total': Decimal('15.10'), 'suicidal_pct': Decimal('95.91')}
{'grp': 'prevention (preventive high risk + watchlist)', 'n': 3274, 'pct_of_total': Decimal('11.75'), 'suicidal_pct': Decimal('56.11')}


##### **5) Where does the scoring system fail? (blind spots + early warning) (bi-variant)**

In [9]:
# Blind spots vs Early warning: count + suicidal%
sql = f"""
SELECT
  grp,
  n,
  ROUND(100.0 * suicidal_yes / n, 2) AS suicidal_pct
FROM (
  SELECT
    'hidden risk (depression=1 & red_flag<=2)' AS grp,
    COUNT(*) AS n,
    SUM(CASE WHEN LOWER(TRIM(suicidal_thoughts))='yes' THEN 1 ELSE 0 END) AS suicidal_yes
  FROM {TABLE}
  WHERE depression = 1 AND red_flag <= 2

  UNION ALL

  SELECT
    'preventive high risk (depression=0 & red_flag>=5)' AS grp,
    COUNT(*) AS n,
    SUM(CASE WHEN LOWER(TRIM(suicidal_thoughts))='yes' THEN 1 ELSE 0 END) AS suicidal_yes
  FROM {TABLE}
  WHERE depression = 0 AND red_flag >= 5
) t;
"""
for r in fetch_all(sql):
    print(r)


{'grp': 'hidden risk (depression=1 & red_flag<=2)', 'n': 2803, 'suicidal_pct': Decimal('67.14')}
{'grp': 'preventive high risk (depression=0 & red_flag>=5)', 'n': 192, 'suicidal_pct': Decimal('77.60')}


In [10]:
sql = f"""
SELECT
  grp,
  n,
  ROUND(100.0 * depressed / n, 2) AS depression_pct,
  ROUND(100.0 * suicidal_yes / n, 2) AS suicidal_pct
FROM (
  SELECT
    'hidden risk (depression=1 & red_flag<=2)' AS grp,
    COUNT(*) AS n,
    SUM(CASE WHEN depression=1 THEN 1 ELSE 0 END) AS depressed,
    SUM(CASE WHEN LOWER(TRIM(suicidal_thoughts))='yes' THEN 1 ELSE 0 END) AS suicidal_yes
  FROM {TABLE}
  WHERE depression = 1 AND red_flag <= 2

  UNION ALL

  SELECT
    'preventive high risk (depression=0 & red_flag>=5)' AS grp,
    COUNT(*) AS n,
    SUM(CASE WHEN depression=1 THEN 1 ELSE 0 END) AS depressed,
    SUM(CASE WHEN LOWER(TRIM(suicidal_thoughts))='yes' THEN 1 ELSE 0 END) AS suicidal_yes
  FROM {TABLE}
  WHERE depression = 0 AND red_flag >= 5
) t;
"""
for r in fetch_all(sql):
    print(r)


{'grp': 'hidden risk (depression=1 & red_flag<=2)', 'n': 2803, 'depression_pct': Decimal('100.00'), 'suicidal_pct': Decimal('67.14')}
{'grp': 'preventive high risk (depression=0 & red_flag>=5)', 'n': 192, 'depression_pct': Decimal('0.00'), 'suicidal_pct': Decimal('77.60')}
